In [ ]:
import numpy as np
from qubicpack.qubicfp import qubicfp
import sys,os
import glob
import yaml
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab

In [ ]:
day = '2025-05-23'
data_dir = '/home/qubic/Calib-TD/'+day+'/'
words = ['Skydip']
keywords = ['*{}*'.format(word) for word in words]
for keyword in keywords:
    dirs = np.sort(glob.glob(data_dir+keyword))
    print(dirs)

In [ ]:
data_path = "/home/belen/Doctorado/qubic-dev/qubic/qubic/data/Flux_jumps/"
soft_path = "/home/belen/Doctorado/qubic-dev/qubic/qubic/lib/Calibration/"
dict_path = "/home/belen/Doctorado/qubic-dev/qubic/qubic/dicts/"
sys.path.append(os.path.abspath(soft_path))

In [ ]:
import Qfluxjumps as jr
import Qsaturation as qsat

## 15.26.08 Skydip

In [ ]:
dataset0 = dirs[2]
a2 = qubicfp()
a2.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a2.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation(sat_value=4.19e6, TES_number=256) 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2025.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
iv_idx = np.where(iv_mask == False)[0]
iv_tod = todarray[iv_idx, :]

In [ ]:
iv_idx  # TES with bad IV curves in serveral datasets

In [ ]:
# Step 1: flux jump detection

In [ ]:
thr = 3e5  # Multiple thresholds for better detection
window_size = 500  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}


In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
len(TES_yes)

In [ ]:
TES_yes

In [ ]:
jr.plot_no_jumps(tt, todarray, results)

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=1e3, thr_amp=2e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)
    


In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
jr.plot_corrections(tt, todarray, results, DT = True)

In [ ]:
jr.plot_corrections(tt, todarray, results, DT = False)

In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=False, use_corrected=True
)

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
metrics_haar[137]

In [ ]:
metrics_dt[137]

In [ ]:
#plt.plot(tt, todarray[137])
plt.plot(tt, corrected_tod[137], label='with DT')
#plt.plot(tt, corrected_tod_nodt[137], label='without DT')

plt.legend()

In [ ]:
for i in TES_with_dt_jumps:
    plt.plot(abs(np.asarray(offset[i])), marker='s', lw=0, label='TES_{}'.format(i+1))
    plt.legend()
    
plt.axhspan(ymin=3.5e5, ymax=5.5e5, color='grey', alpha=0.3)
plt.axhline(y=4.5e5, color='black')
plt.axhline(y=9e5, color='black')

#plt.ylim(2e5, 1.5e6)
plt.ylabel('amplitude')
plt.xlabel('# jumps found')

In [ ]:
results

In [ ]:
results.keys()

In [ ]:
jr.save_results(results, output_dir=data_path + "year_25_day_2305_15.26.08", dataset_name="2305_15.26.08")

## 19.34.50 skydip

In [ ]:
dataset0 = dirs[3]
a3 = qubicfp()
a3.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a3.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60 # 1 hora

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation() 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2025.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
iv_idx = np.where(iv_mask == False)[0]
iv_tod = todarray[iv_idx, :]
iv_idx

In [ ]:
thr = 2e5  # Multiple thresholds for better detection
window_size = 300  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}


In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
len(TES_yes)

In [ ]:
jr.plot_no_jumps(tt, todarray, results)

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=1e3, thr_amp=2e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
jr.plot_corrections(tt, todarray, results)

In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=False, use_corrected=True
)

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
for i in TES_with_dt_jumps:
    plt.plot(abs(np.asarray(offset[i])), marker='s', lw=0, label='TES_{}'.format(i+1))
    plt.legend()
    
plt.axhspan(ymin=3.5e5, ymax=5.5e5, color='grey', alpha=0.3)
plt.axhline(y=4.5e5, color='black')
plt.axhline(y=9e5, color='black')

#plt.ylim(2e5, 1.5e6)
plt.ylabel('amplitude')
plt.xlabel('# jumps found')

In [ ]:
jr.save_results(results, output_dir=data_path + "year_25_day_2305_19.34.50", dataset_name="2305_19.34.50")

## 23.03.55 Skydip

In [ ]:
dataset0 = dirs[4]
a4 = qubicfp()
a4.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a4.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation() 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2025.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
iv_idx  = np.where(iv_mask==False)[0]
iv_idx

In [ ]:
thr = 2e5  # Multiple thresholds for better detection
window_size = 500  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}

In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
len(TES_yes)

In [ ]:
jr.plot_no_jumps(tt, todarray, results)

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=0.5e3, thr_amp=2e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

Results without DT are already well enough

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
jr.plot_corrections(tt, todarray, results, DT=True)

In [ ]:
jr.plot_corrections(tt, todarray, results)

In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=False, use_corrected=True
)

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
jr.save_results(results, output_dir=data_path + "year_25_day_2305_23.03.55", dataset_name="2305_23.03.55")

## 23.19.11 Skydip

In [ ]:
dataset0 = dirs[5]
a5 = qubicfp()
a5.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a5.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation(sat_value=4.19e6, TES_number=256) 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2025.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
thr = 2e5  # Multiple thresholds for better detection
window_size = 250  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}

In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
len(TES_yes)

In [ ]:
jr.plot_no_jumps(tt, todarray, results)

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=0.5e3, thr_amp=2e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

Results without DT are already well enough

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt


In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=2000, robust=False, use_dt=False, use_corrected=True
)

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
metrics_haar[174]

In [ ]:
metrics_dt[174]

In [ ]:
jr.plot_corrections(tt, todarray, results, DT=True)

In [ ]:
jr.plot_corrections(tt, todarray, results)

In [ ]:
jr.save_results(results, output_dir=data_path + "year_25_day_2305_23.19.11", dataset_name="2305_23.19.11")

## 23.24.47 Skydip

In [ ]:
dataset0 = dirs[6]
a6 = qubicfp()
a6.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a6.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation(sat_value=4.19e6, TES_number=256) 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2025.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
thr = 2e5  # Multiple thresholds for better detection
window_size = 300  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}

In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
len(TES_yes)

In [ ]:
jr.plot_no_jumps(tt, todarray, results)

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=1e3, thr_amp=2e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
jr.plot_corrections(tt, todarray, results)

In [ ]:
jr.save_results(results, output_dir=data_path + "year_25_day_2305_23.24.47", dataset_name="2305_23.24.47")